In [0]:
# Exercise 11: Dimensional Modeling for Drone Radiation Data
# first of all we load the source data from S3

from pyspark.sql import functions as F

SOURCE = "s3a://data5035-spring26/drone_data.json"

source_df = spark.read.json(SOURCE)
display(source_df.limit(10))

print(f"Loaded {source_df.count()} records from source data")
print(f"Columns: {', '.join(source_df.columns)}")


# Dimensional Model Design for Drone Radiation Data

## Star Schema Overview
This dimensional model uses a star schema with one fact table and three dimension tables to analyze radiation readings collected by drone

---

##  list of 3 Dimension tables

### 1. **dim_detector** (Detector Dimension) - SCD Type 2
**Description:** Contains information about all radiation detection equipment used in data collection. Consolidates GPS units, Gamma , Cesium  and Thorium detectors into a single dimension

**Type:** SCD Type 2 - tracks historical changes in calibration precision

### 2. **dim_location** (Location Dimension)
**Description:** Contains unique GPS coordinates where drone collected radiation measurements.


### 3. **dim_time** (Time Dimension)
**Description:** Contains timestamps of when radiation readings were collected, broken down into date/time components.

## Fact Table

### **fact_radiation** (Radiation Measurements Fact)
**Description:** Contains actual radiation level measurements with foreign keys to dimensions.


In [0]:
# Dimension Table 1,  detcetors (SCD Type 2)
# Track historical changes in calibration_precision over time

from pyspark.sql.functions import col, to_timestamp, monotonically_increasing_id, lit, lead, coalesce, when
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BooleanType

print("Creating dim_detector with SCD Type 2...")

# Callect all detector calibration records from source data
detector_records = []

# GPS detectors
gps_data = source_df.select(
    col('GPS_UNIT_NUMBER').alias('detector_unit_number'),
    lit('GPS').alias('detector_type'),
    col('GPS_UNIT_CALIBRATION_PRECISION').alias('calibration_precision'),
    to_timestamp(col('GPS_UNIT_CALIBRATION_TIMESTAMP')).alias('calibration_timestamp')
).distinct()

# Gamma detectors
gamma_data = source_df.select(
    col('GAMMA_DETECTOR_UNIT_NUMBER').alias('detector_unit_number'),
    lit('GAMMA').alias('detector_type'),
    col('GAMMA_DETECTOR_CALIBRATION_PRECISION').alias('calibration_precision'),
    to_timestamp(col('GAMMA_DETECTOR_CALIBRATION_TIMESTAMP')).alias('calibration_timestamp')
).distinct()

# cesium detector
cesium_data = source_df.select(
    col('CESIUM_137_DETECTOR_UNIT_NUMBER').alias('detector_unit_number'),
    lit('CESIUM_137').alias('detector_type'),
    col('CESIUM_137_DETECTOR_CALIBRATION_PRECISION').alias('calibration_precision'),
    to_timestamp(col('CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP')).alias('calibration_timestamp')
).distinct()

# Thorium detectors
thorium_data = source_df.select(
    col('THORIUM_232_DETECTOR_UNIT_NUMBER').alias('detector_unit_number'),
    lit('THORIUM_232').alias('detector_type'),
    col('THORIUM_232_DETECTOR_CALIBRATION_PRECISION').alias('calibration_precision'),
    to_timestamp(col('THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP')).alias('calibration_timestamp')
).distinct()

# Union all detector types
all_detectors = gps_data.union(gamma_data).union(cesium_data).union(thorium_data)

# Define window for SCD Type 2: partition by detector, order by calibration time
window_spec = Window.partitionBy('detector_unit_number', 'detector_type').orderBy('calibration_timestamp')

# Implement SCD Type 2 logic:
# - effective_start_date = calibration_timestamp
# - effective_end_date = next calibration_timestamp 
# - is_current = True if it's the last/most recent calibration

dim_detector = all_detectors \
    .withColumn('effective_start_date', col('calibration_timestamp')) \
    .withColumn('next_calibration_timestamp', lead('calibration_timestamp').over(window_spec)) \
    .withColumn('effective_end_date', 
        coalesce(
            col('next_calibration_timestamp'),
            lit('9999-12-31 23:59:59').cast('timestamp')
        )
    ) \
    .withColumn('is_current', 
        when(col('next_calibration_timestamp').isNull(), True).otherwise(False)
    ) \
    .drop('calibration_timestamp', 'next_calibration_timestamp')

# add surrogate key (detector_key) as primary key
dim_detector = dim_detector.withColumn('detector_key', monotonically_increasing_id())

# reorder columns for clarity
dim_detector = dim_detector.select(
    'detector_key',
    'detector_unit_number',
    'detector_type',
    'calibration_precision',
    'effective_start_date',
    'effective_end_date',
    'is_current'
)

# save to fact table
dim_detector.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_detector")

print(f"✓ Saved dim_detector table with SCD Type 2: {dim_detector.count()} rows")
print(f"  (includes historical calibration changes for each detector)")

# Show sample - sort by detector and effective dates to see history
display(dim_detector.orderBy('detector_unit_number', 'effective_start_date'))

In [0]:
# Dimension Table 2, Location
# Create a dimension for unique GPS locations and then save to fact table

from pyspark.sql.functions import monotonically_increasing_id

print("Creating dim_location...")

dim_location = source_df.select(
    'GPS_LAT',
    'GPS_LNG'
).distinct().withColumn('location_id', monotonically_increasing_id())

# save to table
dim_location.write.mode("overwrite").saveAsTable("workspace.default.dim_location")

print(f"✓ Saved dim_location table: {dim_location.count()} unique locations")
display(dim_location.limit(10))

In [0]:
# Dimension Table 3, Time
# Create a time dimension from collection timestamps
from pyspark.sql.functions import to_timestamp, hour, dayofmonth, month, year, dayofweek, date_format

print("Creating dim_time...")

dim_time = source_df.select(
    to_timestamp('COLLECTION_TIMESTAMP').alias('timestamp')
).distinct()

# Add time attributes
dim_time = dim_time.withColumn('date', date_format('timestamp', 'yyyy-MM-dd')) \
    .withColumn('hour', hour('timestamp')) \
    .withColumn('day', dayofmonth('timestamp')) \
    .withColumn('month', month('timestamp')) \
    .withColumn('year', year('timestamp')) \
    .withColumn('day_of_week', dayofweek('timestamp')) \
    .withColumn('time_id', monotonically_increasing_id())

# and then here save to table
dim_time.write.mode("overwrite").saveAsTable("workspace.default.dim_time")

print(f"✓ Saved dim_time table: {dim_time.count()} unique timestamps")
display(dim_time.orderBy('timestamp').limit(10))

In [0]:
# it is Fact Table: Radiation Readings
# Join source data with dimension tables to create fact table with foreign keys


from pyspark.sql.functions import to_timestamp, monotonically_increasing_id, col

print("Creating fact_radiation...")

# Load saved dimension tables (to ensure we use the saved versions)
dim_location = spark.table("workspace.default.dim_location")
dim_time = spark.table("workspace.default.dim_time")
dim_detector = spark.table("workspace.default.dim_detector")

# Start with source data
fact_radiation = source_df

# add timestamp for joining with time dimension
fact_radiation = fact_radiation.withColumn(
    'timestamp', 
    to_timestamp('COLLECTION_TIMESTAMP')
)

# Join with location dimension to get location_id
fact_radiation = fact_radiation.join(
    dim_location,
    (fact_radiation.GPS_LAT == dim_location.GPS_LAT) & 
    (fact_radiation.GPS_LNG == dim_location.GPS_LNG),
    'left'
).drop(dim_location.GPS_LAT).drop(dim_location.GPS_LNG)

# join with time dimension to get time_id
fact_radiation = fact_radiation.join(
    dim_time,
    fact_radiation.timestamp == dim_time.timestamp,
    'left'
).drop(dim_time.timestamp).drop(dim_time.date).drop(dim_time.hour).drop(dim_time.day).drop(dim_time.month).drop(dim_time.year).drop(dim_time.day_of_week)

# join with detector dimension SCD Type 2 to get detector_key
# Need to join 4 times - once for each detector type (GPS, GAMMA, CESIUM_137, THORIUM_232)
# Join condition, unit_number matches and timestamp is between effective dates

# GPS detector join
fact_radiation = fact_radiation.join(
    dim_detector.filter(col('detector_type') == 'GPS').alias('gps_det'),
    (fact_radiation.GPS_UNIT_NUMBER == col('gps_det.detector_unit_number')) &
    (fact_radiation.timestamp >= col('gps_det.effective_start_date')) &
    (fact_radiation.timestamp < col('gps_det.effective_end_date')),
    'left'
).withColumnRenamed('detector_key', 'gps_detector_key').drop('detector_unit_number', 'detector_type', 'calibration_precision', 'effective_start_date', 'effective_end_date', 'is_current')

# Gamma detector join
fact_radiation = fact_radiation.join(
    dim_detector.filter(col('detector_type') == 'GAMMA').alias('gamma_det'),
    (fact_radiation.GAMMA_DETECTOR_UNIT_NUMBER == col('gamma_det.detector_unit_number')) &
    (fact_radiation.timestamp >= col('gamma_det.effective_start_date')) &
    (fact_radiation.timestamp < col('gamma_det.effective_end_date')),
    'left'
).withColumnRenamed('detector_key', 'gamma_detector_key').drop('detector_unit_number', 'detector_type', 'calibration_precision', 'effective_start_date', 'effective_end_date', 'is_current')

# Cesium detector join
fact_radiation = fact_radiation.join(
    dim_detector.filter(col('detector_type') == 'CESIUM_137').alias('cesium_det'),
    (fact_radiation.CESIUM_137_DETECTOR_UNIT_NUMBER == col('cesium_det.detector_unit_number')) &
    (fact_radiation.timestamp >= col('cesium_det.effective_start_date')) &
    (fact_radiation.timestamp < col('cesium_det.effective_end_date')),
    'left'
).withColumnRenamed('detector_key', 'cesium_detector_key').drop('detector_unit_number', 'detector_type', 'calibration_precision', 'effective_start_date', 'effective_end_date', 'is_current')

# thoriom detector join
fact_radiation = fact_radiation.join(
    dim_detector.filter(col('detector_type') == 'THORIUM_232').alias('thorium_det'),
    (fact_radiation.THORIUM_232_DETECTOR_UNIT_NUMBER == col('thorium_det.detector_unit_number')) &
    (fact_radiation.timestamp >= col('thorium_det.effective_start_date')) &
    (fact_radiation.timestamp < col('thorium_det.effective_end_date')),
    'left'
).withColumnRenamed('detector_key', 'thorium_detector_key').drop('detector_unit_number', 'detector_type', 'calibration_precision', 'effective_start_date', 'effective_end_date', 'is_current')

# Select only fact table columns (measurements and foreign keys)
fact_radiation = fact_radiation.select(
    'location_id',
    'time_id',
    'gps_detector_key',
    'gamma_detector_key',
    'cesium_detector_key',
    'thorium_detector_key',
    'GAMMA_LEVEL',
    'CESIUM_137_LEVEL',
    'THORIUM_232_LEVEL',
    'GPS_UNIT_NUMBER',
    'GAMMA_DETECTOR_UNIT_NUMBER',
    'CESIUM_137_DETECTOR_UNIT_NUMBER',
    'THORIUM_232_DETECTOR_UNIT_NUMBER'
).withColumn('reading_id', monotonically_increasing_id())

# finally save all to fact table
fact_radiation.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.fact_radiation")

print(f"✓ Saved fact_radiation table: {fact_radiation.count()} radiation readings")
print("  (with SCD Type 2 detector keys matching effective dates)")
display(fact_radiation.limit(10))

In [0]:
# Verify the dimensional model by loading saved tables and running sample queries
# This show that tables were saved correctly 

from pyspark.sql.functions import col

print("="*60)
print("DIMENSIONAL MODEL VERIFICATION (with SCD Type 2)")
print("="*60)

# Load saved tables
fact_radiation = spark.table("workspace.default.fact_radiation")
dim_location = spark.table("workspace.default.dim_location")
dim_time = spark.table("workspace.default.dim_time")
dim_detector = spark.table("workspace.default.dim_detector")

print("\n=== Dimensional Model Summary ===")
print(f"✓ fact_radiation: {fact_radiation.count():,} rows")
print(f"✓ dim_location: {dim_location.count():,} rows")
print(f"✓ dim_time: {dim_time.count():,} rows")
print(f"✓ dim_detector: {dim_detector.count():,} rows (with historical calibrations)")

print("\n=== SCD Type 2 Verification: Detector History ===")
print("Showing calibration history for GPS-001:\n")

gps_history = dim_detector.filter(col('detector_unit_number') == 'GPS-001') \
    .orderBy('effective_start_date')
display(gps_history)

print("\n=== Current vs Historical Detectors ===")
print(f"Current (active) detectors: {dim_detector.filter(col('is_current') == True).count()}")
print(f"Historical (expired) detectors: {dim_detector.filter(col('is_current') == False).count()}")

print("\n=== Sample Query: Join Fact with Dimensions ===")
print("Query: Show radiation readings with location, time, and detector details\n")

sample_query = fact_radiation.join(
    dim_location,
    'location_id',
    'left'
).join(
    dim_time,
    'time_id',
    'left'
).join(
    dim_detector.alias('gamma_det'),
    fact_radiation.gamma_detector_key == col('gamma_det.detector_key'),
    'left'
).select(
    'reading_id',
    dim_location.GPS_LAT,
    dim_location.GPS_LNG,
    dim_time.date,
    dim_time.hour,
    col('gamma_det.detector_unit_number').alias('gamma_detector'),
    col('gamma_det.calibration_precision').alias('gamma_calibration'),
    col('gamma_det.is_current').alias('gamma_is_current'),
    'GAMMA_LEVEL',
    'CESIUM_137_LEVEL',
    'THORIUM_232_LEVEL'
).limit(20)

display(sample_query)

print("\n" + "="*60)
print("✓ ALL TABLES SAVED SUCCESSFULLY!")
print("✓ Dimensional model with SCD Type 2 is complete and queryable.")
print("✓ Each radiation reading is linked to the correct detector")
print("  calibration that was active at the time of measurement.")
print("="*60)


## Summary

in this section show why we built the detector table this way and what problems it solves

### The Basic Problem

Detectors get recalibrated over time. When we recalibrate a detector, the precision number changes. For example:
- March 28: GPS-001 has precision 1.5
- April 1: They recalibrate it, now precision is 1.2
- April 3: They recalibrate again, now precision is 0.9

If we just kept one row per detector, we will only know the current value (0.9) and willvlose all the old values. This causes problems when analyzing radiation data collected in the past and now we have the complete history.

### Summary

SCD Type 2 lets us keep history instead of overwriting old data. This is important for any dimension where values change over time and we need to remember what the old values were. In our case, we need to remember old calibration precisions so we can correctly analyze radiation measurements from the past.

All tables are saved in databrick > workspace > default as:
- workspace.default.dim_detector 
- workspace.default.dim_location 
- workspace.default.dim_time 
- workspace.default.fact_radiation 

